# Polyfish ML Training on Google Colab Pro (A100)

This notebook sets up the Rust environment and builds Polyfish with CUDA support for training on A100 GPUs.

**Requirements:**
- Google Colab Pro (for A100 GPU access)
- Runtime type: A100 GPU

---

## 1. Verify GPU Type

Make sure you're running on an **A100 GPU** (compute capability 8.0+) which supports BFloat16 tensor cores.

In [1]:
!nvidia-smi
print("\n" + "="*80 + "\n")
!nvcc --version

Wed Feb  4 00:05:37 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.54.15              Driver Version: 550.54.15      CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   38C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## 2. Clone Repository

Clone your Polyfish repository from GitHub.

In [ ]:
!git clone https://github.com/HenBOMB/Polyfish.git /content/Polyfish
%cd /content/Polyfish/polyfish-rs

## 3. Install Rust

Install the Rust toolchain with default settings.

In [ ]:
%%bash
# Install Rust
curl --proto '=https' --tlsv1.2 -sSf https://sh.rustup.rs | sh -s -- -y

# Source the cargo environment
source $HOME/.cargo/env

# Verify installation
rustc --version
cargo --version

## 4. Configure CUDA Environment

Set up CUDA paths and compute capability for A100 (sm_80).

In [ ]:
import os

# Set CUDA environment variables
os.environ['CUDA_HOME'] = '/usr/local/cuda'
os.environ['CUDA_ROOT'] = '/usr/local/cuda'
os.environ['LD_LIBRARY_PATH'] = f"{os.environ.get('CUDA_HOME', '')}/lib64:{os.environ.get('LD_LIBRARY_PATH', '')}"
os.environ['PATH'] = f"{os.environ.get('CUDA_HOME', '')}/bin:{os.environ.get('PATH', '')}"

# A100 compute capability
os.environ['CUDA_COMPUTE_CAP'] = '80'
os.environ['TORCH_CUDA_ARCH_LIST'] = '8.0'

print("CUDA Environment:")
print(f"CUDA_HOME: {os.environ.get('CUDA_HOME')}")
print(f"CUDA_COMPUTE_CAP: {os.environ.get('CUDA_COMPUTE_CAP')}")
print(f"\nLD_LIBRARY_PATH: {os.environ.get('LD_LIBRARY_PATH')}")

## 5. Build with CUDA Support

Build the Polyfish engine with CUDA and cuDNN features enabled.

In [ ]:
%%bash
source $HOME/.cargo/env

# Set CUDA environment for build
export CUDA_HOME=/usr/local/cuda
export CUDA_ROOT=/usr/local/cuda
export LD_LIBRARY_PATH=$CUDA_HOME/lib64:$LD_LIBRARY_PATH
export PATH=$CUDA_HOME/bin:$PATH
export CUDA_COMPUTE_CAP=80

# Build with CUDA support
cd /content/Polyfish/polyfish-rs
cargo build --release --features cuda

echo "\n========================================"
echo "Build complete!"
echo "========================================"

## 6. Run Training

Execute the training binary with CUDA acceleration.

In [ ]:
%%bash
source $HOME/.cargo/env

# Set runtime CUDA environment
export CUDA_HOME=/usr/local/cuda
export LD_LIBRARY_PATH=$CUDA_HOME/lib64:$LD_LIBRARY_PATH

cd /content/Polyfish/polyfish-rs

# Run your training command here
# Example: cargo run --release --features cuda --bin train
./target/release/polyfish-rs

## 7. Monitor GPU Usage

Check GPU utilization during training.

In [ ]:
# Run this cell while training is running in another cell
!nvidia-smi --query-gpu=timestamp,name,temperature.gpu,utilization.gpu,utilization.memory,memory.used,memory.total --format=csv -l 5

## 8. Save Model Checkpoints

Mount Google Drive to persist your trained models.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Create checkpoint directory
!mkdir -p /content/drive/MyDrive/polyfish_checkpoints

# Copy checkpoints to Drive (example)
# !cp /content/Polyfish/polyfish-rs/checkpoints/* /content/drive/MyDrive/polyfish_checkpoints/

## 9. Quick Test (Optional)

Run a quick CUDA test to verify everything is working.

In [ ]:
%%bash
source $HOME/.cargo/env

export CUDA_HOME=/usr/local/cuda
export LD_LIBRARY_PATH=$CUDA_HOME/lib64:$LD_LIBRARY_PATH

cd /content/Polyfish/polyfish-rs

# Run tests with CUDA
cargo test --release --features cuda -- --nocapture